In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece xgboost

In [ ]:
import numpy as np
import pandas as pd
import torch
import os
import zipfile
from datasets import load_dataset, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' 

In [ ]:
MODEL_NAME = "FacebookAI/roberta-large"
MAX_LENGTH = 512
N_FOLDS = 7

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
id2label = {v: k for k, v in label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': accuracy, 'f1': f1}

def tokenize_function(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    if "clarity_label" in examples:
        tokenized["labels"] = [label2id[l] for l in examples["clarity_label"]]
    return tokenized

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
ds = load_dataset("ailsntua/QEvasion")
train_full_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

In [ ]:
train_full_dataset = Dataset.from_pandas(train_full_df, preserve_index=False)
train_tokenized_full = train_full_dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=train_full_dataset.column_names
)

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_tokenized = test_dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=[col for col in test_dataset.column_names if col not in ['clarity_label']]
)

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_test_probs = []    
fold_oof_probs = []   
fold_metrics = []


for fold, (train_idx, val_idx) in enumerate(skf.split(train_full_df, train_full_df['clarity_label']), 1):
    print(f"\nFold {fold}/{N_FOLDS}")
    print(f"{'='*30}")
    
    train_dataset = train_tokenized_full.select(train_idx)
    val_dataset = train_tokenized_full.select(val_idx)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label=id2label,
        label2id=label2id
    )
    
    training_args = TrainingArguments(
        output_dir=f"./results_roberta_fold{fold}",
        learning_rate=1e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=15,
        warmup_ratio=0.1,
        weight_decay=0.01,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        bf16=True,
        bf16_full_eval=True,
        optim="adamw_torch",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        seed=SEED + fold, 
        report_to="none",
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1")],
    )

    trainer.train()

    val_results = trainer.evaluate()
    fold_metrics.append({
        'fold': fold,
        'val_accuracy': val_results['eval_accuracy'],
        'val_f1': val_results['eval_f1']
    })
    
    print(f"Fold {fold} Result: Accuracy={val_results['eval_accuracy']:.4f}, F1={val_results['eval_f1']:.4f}")
    
    oof_output = trainer.predict(val_dataset)
    oof_probs = torch.nn.functional.softmax(torch.tensor(oof_output.predictions), dim=-1).numpy()
    fold_oof_probs.append(oof_probs)
    fold_oof_indices.append(val_idx)

    test_output = trainer.predict(test_tokenized)
    test_probs = torch.nn.functional.softmax(torch.tensor(test_output.predictions), dim=-1).numpy()
    fold_test_probs.append(test_probs)

    del model, trainer
    torch.cuda.empty_cache()


In [ ]:

print("Cross-Validation Metrics")

metrics_df = pd.DataFrame(fold_metrics)
print(f"\nAverage Validation Accuracy: {metrics_df['val_accuracy'].mean():.4f} ± {metrics_df['val_accuracy'].std():.4f}")
print(f"Average Validation Macro F1: {metrics_df['val_f1'].mean():.4f} ± {metrics_df['val_f1'].std():.4f}")

oof_probs_full = np.zeros((len(train_full_df), 3))
for probs, idxs in zip(fold_oof_probs, fold_oof_indices):
    oof_probs_full[idxs] = probs

print(f"OOF probability matrix shape: {oof_probs_full.shape}")
print(f"  {len(train_full_df)} training samples x 3 class probabilities")

oof_preds_base = np.argmax(oof_probs_full, axis=1)
oof_labels_base = [id2label[p] for p in oof_preds_base]
y_true_train = train_full_df['clarity_label'].tolist()
y_true_train_numeric = [label2id[label] for label in y_true_train]

oof_acc_base = accuracy_score(y_true_train, oof_labels_base)
oof_f1_base = f1_score(y_true_train, oof_labels_base, average='macro', labels=clarity_labels)

print(f"\nBase Model OOF Performance (simple averaging):")
print(f"  Accuracy: {oof_acc_base:.4f}")
print(f"  Macro F1: {oof_f1_base:.4f}")

In [ ]:

print("Training XGBoost Stacker")

xgb_stacker = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    eval_metric='mlogloss',
    tree_method='auto'
)

xgb_stacker.fit(oof_probs_full, y_true_train_numeric)

oof_preds_stacked = xgb_stacker.predict(oof_probs_full)
oof_labels_stacked = [id2label[p] for p in oof_preds_stacked]

oof_acc_stacked = accuracy_score(y_true_train, oof_labels_stacked)
oof_f1_stacked = f1_score(y_true_train, oof_labels_stacked, average='macro', labels=clarity_labels)

print(f"\nStacked Model OOF Performance:")
print(f"  Accuracy: {oof_acc_stacked:.4f} (Base: {oof_acc_base:.4f}, Δ{oof_acc_stacked-oof_acc_base:+.4f})")
print(f"  Macro F1: {oof_f1_stacked:.4f} (Base: {oof_f1_base:.4f}, Δ{oof_f1_stacked-oof_f1_base:+.4f})")

print("\nOOF Classification Report (Stacked):")
print(classification_report(y_true_train, oof_labels_stacked, labels=clarity_labels, zero_division=0))

print("\nOOF Confusion Matrix (Stacked):")
print(confusion_matrix(y_true_train, oof_labels_stacked, labels=clarity_labels))

In [ ]:
print("Test set predictions")

test_probs_avg = np.mean(fold_test_probs, axis=0)
print(f"Averaged test probabilities shape: {test_probs_avg.shape}")
print(f"  {len(test_df)} test samples x 3 class probabilities")

test_preds_base = np.argmax(test_probs_avg, axis=1)
test_labels_base = [id2label[p] for p in test_preds_base]

y_true_test = test_df['clarity_label'].tolist()

test_acc_base = accuracy_score(y_true_test, test_labels_base)
test_f1_base = f1_score(y_true_test, test_labels_base, average='macro', labels=clarity_labels)

print(f"\nBase Model Test Performance (simple averaging):")
print(f"  Accuracy: {test_acc_base:.4f}")
print(f"  Macro F1: {test_f1_base:.4f}")

test_preds_stacked = xgb_stacker.predict(test_probs_avg)
test_labels_stacked = [id2label[p] for p in test_preds_stacked]

test_acc_stacked = accuracy_score(y_true_test, test_labels_stacked)
test_f1_stacked = f1_score(y_true_test, test_labels_stacked, average='macro', labels=clarity_labels)

print(f"\nStacked Model Test Performance:")
print(f"  Accuracy: {test_acc_stacked:.4f} (Base: {test_acc_base:.4f}, Δ{test_acc_stacked-test_acc_base:+.4f})")
print(f"  Macro F1: {test_f1_stacked:.4f} (Base: {test_f1_base:.4f}, Δ{test_f1_stacked-test_f1_base:+.4f})")

print("\nTest Classification Report (Stacked):")
print(classification_report(y_true_test, test_labels_stacked, labels=clarity_labels, zero_division=0))

print("\nTest Confusion Matrix (Stacked):")
print(confusion_matrix(y_true_test, test_labels_stacked, labels=clarity_labels))

In [ ]:
os.makedirs("./predictions", exist_ok=True)
prediction_filename = "./predictions/prediction"

with open(prediction_filename, 'w') as f:
    for label in test_labels_stacked:
        f.write(f"{label}\n")

zip_filename = "./predictions/prediction"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(prediction_filename, arcname="prediction")